# Suraksha - RAG Pipeline for RBI Guidelines

**Retrieval-Augmented Generation for fraud explanation**

Shared Component - Used by both advanced and baseline solutions

## Purpose:
- Load RBI fraud prevention guidelines (PDF documents)
- Chunk documents into semantic segments
- Generate embeddings using all-MiniLM-L6-v2
- Build FAISS vector index for fast retrieval
- Provide fraud explanations grounded in regulatory guidelines

## Output:
- FAISS index saved to `/dbfs/suraksha/rag/faiss_index/`
- Text chunks saved to `/dbfs/suraksha/rag/chunks.pkl`
- Ready for use by rag_utils for fraud explanations

## Fraud Types Covered:
1. Velocity Fraud
2. Mule Account
3. SIM Swap
4. Device Takeover
5. Beneficiary Manipulation
6. Amount Anomaly
7. Temporal Anomaly
8. Merchant Fraud
9. Failed-Then-Success

In [0]:
# Quick install without version checks to save time
print("📊 Installing RAG pipeline dependencies...")
print("   This takes 5-10 minutes but only needs to run once.\n")

%pip install sentence-transformers faiss-cpu PyPDF2 --quiet

print("\n✓ Packages installed!")
print("🔄 Restarting Python to load packages...\n")

dbutils.library.restartPython()

In [0]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
import pickle
import os
import warnings
warnings.filterwarnings('ignore')

print("="*70)
print("🚀 SURAKSHA - RAG PIPELINE BUILDER")
print("="*70)
print("\n✓ Sentence Transformers imported")
print("✓ FAISS imported")
print("✓ Ready to build RAG index")

In [0]:
print("\n📊 Step 1: Loading RBI documents from PDFs...")

import PyPDF2
import re
import os

# Get current user dynamically
username = dbutils.notebook.entry_point.getDbutils().notebook().getContext().userName().get()
rbi_docs_path = f"/Workspace/Users/{username}/Suraksha/data/rbi_documents"

# Load all PDFs
all_pdf_text = []
pdf_metadata = []

print(f"\n📄 Processing PDFs from: {rbi_docs_path}")
print(f"   (User: {username})\n")

pdf_files = [f for f in os.listdir(rbi_docs_path) if f.endswith('.pdf')]

for pdf_file in pdf_files:
    pdf_path = os.path.join(rbi_docs_path, pdf_file)
    try:
        with open(pdf_path, 'rb') as file:
            pdf_reader = PyPDF2.PdfReader(file)
            num_pages = len(pdf_reader.pages)
            
            # Extract text from all pages
            pdf_text = ""
            for page_num in range(num_pages):
                page = pdf_reader.pages[page_num]
                pdf_text += page.extract_text() + "\n"
            
            all_pdf_text.append(pdf_text)
            pdf_metadata.append({
                'filename': pdf_file,
                'pages': num_pages,
                'length': len(pdf_text)
            })
            
            print(f"✓ {pdf_file}: {num_pages} pages, {len(pdf_text):,} chars")
    
    except Exception as e:
        print(f"✗ Error reading {pdf_file}: {str(e)}")

# Combine all PDF text
combined_rbi_text = "\n\n".join(all_pdf_text)
print(f"\n✓ Total documents loaded: {len(all_pdf_text)}")
print(f"✓ Total text extracted: {len(combined_rbi_text):,} characters")

# Create fast-lookup templates for fraud types
# These templates map fraud types to key search terms in RBI documents
fraud_type_templates = {
    "Velocity Fraud": {
        "keywords": ["velocity", "rapid transaction", "multiple transaction", "frequency", "transaction limit"],
        "quick_desc": "Multiple rapid transactions in quick succession to maximize stolen funds before detection.",
        "rbi_section": "Transaction Monitoring and Velocity Controls"
    },
    "Mule Account": {
        "keywords": ["money mule", "money laundering", "AML", "PMLA", "structuring", "layering"],
        "quick_desc": "Accounts used to launder stolen funds by receiving and immediately transferring money.",
        "rbi_section": "Anti-Money Laundering Guidelines"
    },
    "SIM Swap": {
        "keywords": ["SIM swap", "SIM card", "OTP", "mobile number", "authentication"],
        "quick_desc": "Fraudsters transfer victim's mobile number to gain access to OTPs and banking alerts.",
        "rbi_section": "Mobile Banking Security Guidelines"
    },
    "Device Takeover": {
        "keywords": ["device", "malware", "phishing", "credential", "device fingerprint"],
        "quick_desc": "Unauthorized access to victim's device through malware or stolen credentials.",
        "rbi_section": "Device Security and Authentication"
    },
    "Beneficiary Manipulation": {
        "keywords": ["social engineering", "impersonation", "beneficiary", "customer awareness"],
        "quick_desc": "Manipulating users into sending money to fraudulent accounts via impersonation.",
        "rbi_section": "Customer Protection and Awareness"
    },
    "Amount Anomaly": {
        "keywords": ["anomaly", "unusual transaction", "statistical", "threshold", "risk score"],
        "quick_desc": "Transactions significantly deviating from user's typical transaction patterns.",
        "rbi_section": "Transaction Monitoring and Analytics"
    },
    "Temporal Anomaly": {
        "keywords": ["odd hours", "timing", "temporal", "weekend", "holiday"],
        "quick_desc": "Transactions at unusual times when legitimate users are unlikely to transact.",
        "rbi_section": "Behavioral Analytics"
    },
    "Merchant Fraud": {
        "keywords": ["merchant", "point of sale", "POS", "merchant risk", "chargeback"],
        "quick_desc": "Fake merchants, compromised terminals, or merchant-fraudster collusion.",
        "rbi_section": "Merchant Onboarding and Monitoring"
    },
    "Failed-Then-Success": {
        "keywords": ["authentication failure", "brute force", "credential guessing", "MPIN"],
        "quick_desc": "Multiple failed attempts followed by success suggesting credential attacks.",
        "rbi_section": "Authentication Security"
    }
}

print(f"\n✓ Created templates for {len(fraud_type_templates)} fraud types")

# Prepare final text corpus (RBI text + templates)
rbi_guidelines = {}
for fraud_type, template in fraud_type_templates.items():
    # Search for relevant sections in RBI text
    relevant_text = f"**{fraud_type}**\n\n"
    relevant_text += f"Description: {template['quick_desc']}\n\n"
    relevant_text += f"RBI Section: {template['rbi_section']}\n\n"
    
    # Extract relevant paragraphs from combined RBI text based on keywords
    relevant_paragraphs = []
    for keyword in template['keywords']:
        # Find paragraphs containing keywords (case-insensitive)
        pattern = re.compile(f'.{{0,300}}{re.escape(keyword)}.{{0,300}}', re.IGNORECASE | re.DOTALL)
        matches = pattern.findall(combined_rbi_text)
        relevant_paragraphs.extend(matches[:2])  # Top 2 matches per keyword
    
    if relevant_paragraphs:
        relevant_text += "Relevant RBI Guidelines:\n" + "\n\n".join(set(relevant_paragraphs[:5]))
    
    rbi_guidelines[fraud_type] = relevant_text

print(f"\n✓ Built fraud type guidelines with RBI context")
for fraud_type in rbi_guidelines.keys():
    print(f"   - {fraud_type}: {len(rbi_guidelines[fraud_type])} characters")

In [0]:
print("\n📊 Step 2: Chunking documents into semantic segments...")

def chunk_text(text, chunk_size=500, overlap=100):
    """
    Split text into overlapping chunks for better semantic retrieval.
    
    Args:
        text: Text to chunk
        chunk_size: Target size of each chunk in characters
        overlap: Overlap between consecutive chunks
    
    Returns:
        List of text chunks
    """
    chunks = []
    start = 0
    
    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end]
        
        # Try to break at sentence boundary
        if end < len(text):
            last_period = chunk.rfind('.')
            if last_period > chunk_size * 0.7:  # At least 70% through chunk
                end = start + last_period + 1
                chunk = text[start:end]
        
        chunks.append(chunk.strip())
        start = end - overlap
    
    return chunks

# Chunk all guidelines
all_chunks = []
chunk_metadata = []  # Track which fraud type each chunk belongs to

for fraud_type, guideline_text in rbi_guidelines.items():
    chunks = chunk_text(guideline_text)
    all_chunks.extend(chunks)
    chunk_metadata.extend([fraud_type] * len(chunks))
    print(f"✓ {fraud_type}: {len(chunks)} chunks")

print(f"\n✓ Total chunks created: {len(all_chunks)}")
print(f"✓ Chunk metadata entries: {len(chunk_metadata)}")

In [0]:
print("\n📊 Step 3: Loading embedding model and creating embeddings...")

# Load sentence transformer model (all-MiniLM-L6-v2)
print("   Loading all-MiniLM-L6-v2 model...")
model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
print("✓ Model loaded")

# Create embeddings for all chunks
print(f"\n   Creating embeddings for {len(all_chunks)} chunks...")
embeddings = model.encode(all_chunks, show_progress_bar=True, convert_to_numpy=True)

print(f"\n✓ Embeddings created")
print(f"   Shape: {embeddings.shape}")
print(f"   Dimension: {embeddings.shape[1]}")
print(f"   Dtype: {embeddings.dtype}")

In [0]:
print("\n📊 Step 4: Building FAISS index for fast retrieval...")

# Normalize embeddings for cosine similarity
faiss.normalize_L2(embeddings)

# Create FAISS index (using IndexFlatIP for inner product / cosine similarity)
dimension = embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)

# Add embeddings to index
index.add(embeddings)

print(f"✓ FAISS index built")
print(f"   Index type: IndexFlatIP (cosine similarity)")
print(f"   Dimension: {dimension}")
print(f"   Total vectors: {index.ntotal}")

In [0]:
print("\n📊 Step 5: Saving FAISS index and metadata...")

# Get current user dynamically (works for any judge/user)
username = dbutils.notebook.entry_point.getDbutils().notebook().getContext().userName().get()
print(f"Current user: {username}")

# Use dynamic path based on current user
rag_dir = f"/Workspace/Users/{username}/Suraksha/rag/"
index_dir = rag_dir + "faiss_index/"
os.makedirs(index_dir, exist_ok=True)

# Save FAISS index
index_path = index_dir + "index.faiss"
faiss.write_index(index, index_path)
print(f"✓ FAISS index saved: {index_path}")

# Save text chunks
chunks_path = rag_dir + "chunks.pkl"
with open(chunks_path, 'wb') as f:
    pickle.dump(all_chunks, f)
print(f"✓ Text chunks saved: {chunks_path}")

# Save chunk metadata
metadata_path = rag_dir + "metadata.pkl"
with open(metadata_path, 'wb') as f:
    pickle.dump(chunk_metadata, f)
print(f"✓ Chunk metadata saved: {metadata_path}")

# Save embedding model name for reference
model_info = {
    'model_name': 'sentence-transformers/all-MiniLM-L6-v2',
    'dimension': dimension,
    'num_chunks': len(all_chunks),
    'fraud_types': list(rbi_guidelines.keys())
}
model_info_path = rag_dir + "model_info.pkl"
with open(model_info_path, 'wb') as f:
    pickle.dump(model_info, f)
print(f"✓ Model info saved: {model_info_path}")

In [0]:
print("\n📊 Step 6: Testing RAG retrieval...")

# Test query
test_query = "What are the indicators of velocity fraud?"
print(f"\n🔍 Test Query: '{test_query}'")

# Encode query
query_embedding = model.encode([test_query], convert_to_numpy=True)
faiss.normalize_L2(query_embedding)

# Search for top 3 most relevant chunks
k = 3
distances, indices = index.search(query_embedding, k)

print(f"\n🎯 Top {k} Results:\n")
for i, (idx, score) in enumerate(zip(indices[0], distances[0])):
    print(f"Result {i+1} (Score: {score:.4f})")
    print(f"Fraud Type: {chunk_metadata[idx]}")
    print(f"Text: {all_chunks[idx][:200]}...")
    print("-" * 70)

In [0]:
print("\n" + "="*70)
print("✅ RAG PIPELINE BUILD COMPLETE!")
print("="*70)
print("\n📊 Pipeline Summary:")
print(f"   • Fraud types covered: {len(rbi_guidelines)}")
print(f"   • Total text chunks: {len(all_chunks)}")
print(f"   • Embedding dimension: {dimension}")
print(f"   • FAISS index size: {index.ntotal} vectors")
print(f"   • Model: all-MiniLM-L6-v2")

print("\n💾 Saved Artifacts:")
print(f"   • FAISS index: {index_path}")
print(f"   • Text chunks: {chunks_path}")
print(f"   • Metadata: {metadata_path}")
print(f"   • Model info: {model_info_path}")

print("\n🎯 Next Steps:")
print("   1. Use rag_utils.ipynb to load and query this index")
print("   2. Integrate with model serving for fraud explanations")
print("   3. Build demo app with RAG-powered explanations")

print("\n" + "="*70)